In [2]:
import os
import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
csv1 = pd.read_csv("../datasets/Audible_Catlog.csv")
csv1.drop_duplicates(inplace=True)

In [4]:
csv2 = pd.read_csv("../datasets/Audible_Catlog_Advanced_Features.csv")
csv2.drop_duplicates(inplace=True)

In [5]:
def to_hours(text):
            h = re.search(r'(\d+)\s*hour', str(text))
            m = re.search(r'(\d+)\s*minute', str(text))
            hours = int(h.group(1)) if h else 0
            minutes = int(m.group(1)) if m else 0
            return hours + minutes/60
def extract_genres(text):
    if not isinstance(text, str): return []
    
    parts = text.split(',')
    genres = []
    
    for part in parts:
        match = re.search(r'in (.*)', part)
        if match:
            genre = match.group(1).strip()
            genre = re.sub(r'\(.*\)', '', genre).strip()
            if genre and "Audible Audiobooks" not in genre:
                genres.append(genre)
    return genres

In [6]:
df = csv1.merge(csv2, how="inner", on="Book Name")
df = df.drop(['Author_y','Rating_y', 'Price_x'], axis=1)
df['Number of Reviews'] = df[['Number of Reviews_x', 'Number of Reviews_y']].max(axis=1)
df = df.drop(['Number of Reviews_x', 'Number of Reviews_y'], axis=1)
df = df.dropna(subset=['Description'])
df['Number of Reviews'] = df['Number of Reviews'].fillna(0).astype(int)
df = df.copy()
df['Ranks and Genre'] = df['Ranks and Genre'].str.lstrip(',')
df = df.copy()
df['Rating_x'] = df['Rating_x'].replace(-1.0, np.nan)
df['Listening Time'] = df['Listening Time'].replace("-1", "None")
df['Ranks and Genre'] = df['Ranks and Genre'].replace("-1", "None")
df = df.dropna(subset=['Rating_x'])
df = df[df["Listening Time"] != "None"]
df = df[df["Ranks and Genre"] != "None"]
df['Popularity'] = df['Rating_x'] * df['Number of Reviews']
df["Hours"] = df["Listening Time"].apply(to_hours)
df['genre_list'] = df['Ranks and Genre'].apply(extract_genres)
df['main_genre'] = df['genre_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')
df = df[df["main_genre"] != "Unknown"]

df.columns = ['Book Name', 'Author Name', 'Rating', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Number of Reviews', 'Popularity', 'Listening Hours', 'Genre List', 'Main Genre']

In [7]:
df

,Book Name,Author Name,Rating,Price,Description,Listening Time,Ranks and Genre,Number of Reviews,Popularity,Listening Hours,Genre List,Main Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,#1 in Audible Audiobooks & Originals (See Top ...,371,1817.9,10.900000,"[Personal Success, Stress Management, Society ...",Personal Success
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,615,Brought to you by Penguin.,3 hours and 23 minutes,#2 in Audible Audiobooks & Originals (See Top ...,3682,16937.2,3.383333,"[Meditation, Self-Esteem, Personal Success]",Meditation
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,#3 in Audible Audiobooks & Originals (See Top ...,20306,89346.4,5.283333,"[Personal Success, Personal Development & Self...",Personal Success
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,888,Brought to you by Penguin.,5 hours and 35 minutes,#5 in Audible Audiobooks & Originals (See Top ...,4678,21518.8,5.583333,"[Psychology, Stress Management, Personal Success]",Psychology
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,1005,"Stop going through life, Start growing throug...",6 hours and 25 minutes,#6 in Audible Audiobooks & Originals (See Top ...,4308,19816.8,6.416667,"[Literary Essays, Essays, Spiritualism]",Literary Essays
...,...,...,...,...,...,...,...,...,...,...,...,...
3478,"Cruel Prince: Royal Hearts Academy, Book 1",Ashley Jade,4.7,1005,Welcome to their kingdom....,11 hours and 8 minutes,"#4,530 in Romance (Books)",256,1203.2,11.133333,[Romance],Romance
3486,City of Ghosts,Victoria Schwab,4.6,492,From number-one New York Times best-selling au...,5 hours and 2 minutes,"#2,294 in Audible Audiobooks & Originals (See ...",430,1978.0,5.033333,[Paranormal & Supernatural Fantasy for Childre...,Paranormal & Supernatural Fantasy for Children
3498,Think and Grow Rich,Napoleon Hill,4.4,166,\r\nBy completing your purchase you agree to A...,10 hours and 15 minutes,#38 in Audible Audiobooks & Originals (See Top...,22448,98771.2,10.250000,"[Economics, Business Motivation & Self-Improve...",Economics
3508,Terra Incognita: 100 Maps to Survive the Next ...,Dr Ian Goldin,5.0,888,Brought to you by Penguin.,13 hours and 41 minutes,#582 in Audible Audiobooks & Originals (See To...,1,5.0,13.683333,"[Climate Change, Earth Sciences, Future Studies]",Climate Change


In [13]:
df.groupby('Main Genre')['Popularity'].mean()

Main Genre
20th Century Historical Romance                     4639.500000
Absurdist Fiction                                     24.000000
Acting & Auditioning                                1551.600000
Action & Adventure                                 16702.600000
Action & Adventure Anthologies & Short Stories       607.200000
                                                      ...      
World Literature                                     143.500000
World War II Historical Fiction                   196598.400000
World War II History                                2884.266667
Yoga                                                 319.600000
Zen Buddhism                                        2461.200000
Name: Popularity, Length: 619, dtype: float64

In [9]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Description'].fillna(''))
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)